# Tutorial 2 — DMRG and Single-Orbital Entropies

This notebook shows how to configure and run a DMRG calculation with AEGISS,
extract single-orbital entropies from the resulting MPS, and apply the
entropy-based screening step.

**Prerequisites**: `block2` must be installed.  The rest of the AEGISS stack
(PySCF, NumPy) is assumed available.

**Contents**
1. Molecule and mean-field setup
2. DMRGConfig — parameters explained
3. Running DMRG and obtaining entropies
4. Entropy-based orbital screening
5. Visualising single-orbital entropies

## 1. Imports

In [ ]:
import numpy as np

from aegiss import DMRGConfig, EntropyConfig
from aegiss.active_space.entropy_screening import select_by_entropy
from aegiss.initial_guess.scf import build_mol, run_scf
from aegiss.utils.orbital_indices import get_subspaces_from_cas
from aegiss.wavefunction.dmrg import run_dmrg

## 2. Molecule and mean-field setup

We use water (H₂O) for speed.  The same workflow applies to any system —
just adjust `ncas` and `nelecas` in `DMRGConfig` accordingly.

In [ ]:
mol = build_mol(
    atom="O 0 0 0.119; H 0 0.757 -0.477; H 0 -0.757 -0.477",
    basis="sto-3g",
    verbose=0,
)

mf = run_scf(mol, method="RHF")
print(f"RHF energy : {mf.e_tot:.8f} Ha")
print(f"MOs        : {mf.mo_coeff.shape[1]}  (occ: {int(np.sum(mf.mo_occ > 0))})")

## 3. DMRGConfig

`DMRGConfig` defines the CAS and the DMRG solver settings.

| Field | Type | Default | Meaning |
|---|---|---|---|
| `ncas` | `int` | — | Number of active orbitals |
| `nelecas` | `int` or `(int, int)` | — | Active electrons (total or α/β pair) |
| `init_bond_dim` | `int` | `10` | Starting bond dimension; grows during sweeps |
| `symm_type` | `'SZ'` or `'SU2'` | `'SZ'` | Spin symmetry. `'SZ'` required for entropies |
| `verbosity` | `int` | `0` | block2 output level |

> **Note**: `symm_type='SU2'` is faster but cannot produce entropies directly.
> Use `'SZ'` whenever you need the single-orbital entropies.

In [ ]:
# CAS(4e, 4o) — valence space of water
dmrg_config = DMRGConfig(
    ncas=4,
    nelecas=4,
    init_bond_dim=20,
    symm_type="SZ",   # required for entropy output
    verbosity=0,
)
print(dmrg_config)

## 4. Identify the CAS orbital subspaces

`get_subspaces_from_cas` partitions the CAS into occupied (active) and virtual
orbital index arrays using the aufbau occupation from the mean-field object.

In [ ]:
active_orbs, virt_orbs = get_subspaces_from_cas(
    mf,
    ncas=dmrg_config.ncas,
    nelecas=dmrg_config.nelecas,
)
print(f"Active (occ) orbitals : {active_orbs}")
print(f"Active (virt) orbitals: {virt_orbs}")

## 5. Run DMRG

`run_dmrg` launches a block2 DMRG sweep schedule and returns:
- `energy` — the DMRG ground-state energy
- `soe1` — single-orbital entropies $S_1(i)$ for every orbital in the CAS
- `soe2` — two-orbital mutual information matrix
- `minfo` — raw MPS metadata from block2

In [ ]:
energy, soe1, soe2, minfo = run_dmrg(
    mf,
    ncas=dmrg_config.ncas,
    nelecas=dmrg_config.nelecas,
    symm_type=dmrg_config.symm_type,
    init_bond_dim=dmrg_config.init_bond_dim,
    verbosity=dmrg_config.verbosity,
)

print(f"DMRG energy : {energy:.8f} Ha")
print(f"soe1 shape  : {soe1.shape}   (one value per CAS orbital)")
print(f"\nSingle-orbital entropies:")
for idx, s in enumerate(soe1):
    orb = list(active_orbs) + list(virt_orbs)
    print(f"  MO {orb[idx]:>2d}  S1 = {s:.6f}")

## 6. EntropyConfig and orbital screening

`EntropyConfig` controls which orbitals survive the entropy threshold:

| Field | Default | Meaning |
|---|---|---|
| `fraction` | `0.1` | Threshold value (see `fraction_type`) |
| `fraction_type` | `'r'` | `'r'`: keep if $S_1 \geq$ `fraction × max(S_1)`;  `'a'`: keep if $S_1 \geq$ `fraction` |
| `ranking` | `'subset'` | Apply threshold independently per subspace or globally |

In [ ]:
entropy_config = EntropyConfig(
    fraction=0.1,        # keep orbitals with S1 >= 10% of max(S1)
    fraction_type="r",
    ranking="subset",    # threshold applied separately to occ and virt
)
print(entropy_config)

In [ ]:
screened_occ, screened_virt = select_by_entropy(
    occ_orbs=active_orbs,
    virt_orbs=virt_orbs,
    soe1=soe1,
    ranking=entropy_config.ranking,
    fraction=entropy_config.fraction,
    fraction_type=entropy_config.fraction_type,
)

print(f"Entropy-screened occupied : {[o for o, _ in screened_occ]}")
print(f"Entropy-screened virtual  : {[o for o, _ in screened_virt]}")
print(f"\nTotal after screening: {len(screened_occ) + len(screened_virt)} orbitals")

## 7. Tuning the threshold

Increasing `fraction` makes the selection more restrictive; decreasing it is
more inclusive.  The `'global'` ranking pools occupied and virtual orbitals
before thresholding — useful when you expect the most entangled orbitals to
span both spaces.

In [ ]:
for fraction in (0.05, 0.10, 0.20, 0.50):
    s_occ, s_virt = select_by_entropy(
        occ_orbs=active_orbs,
        virt_orbs=virt_orbs,
        soe1=soe1,
        ranking="subset",
        fraction=fraction,
        fraction_type="r",
    )
    n = len(s_occ) + len(s_virt)
    print(f"fraction={fraction:.2f}  →  {n} orbital(s) retained")

## 8. Visualise single-orbital entropies

AEGISS provides `soe_heatmap` and `th_plots` to produce diagnostic PDFs.
Here we do a quick inline bar chart using matplotlib.

In [ ]:
try:
    import matplotlib.pyplot as plt

    all_orbs = list(active_orbs) + list(virt_orbs)
    colors = ["steelblue"] * len(active_orbs) + ["tomato"] * len(virt_orbs)

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(range(len(soe1)), soe1, color=colors)
    ax.axhline(entropy_config.fraction * soe1.max(), ls="--", color="k",
               label=f"threshold ({entropy_config.fraction:.0%} of max)")
    ax.set_xticks(range(len(soe1)))
    ax.set_xticklabels([f"MO {o}" for o in all_orbs], rotation=45)
    ax.set_ylabel("$S_1$")
    ax.set_title("Single-orbital entropies — H₂O CAS(4e,4o)")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plot.")

## Summary

- `DMRGConfig` defines the CAS and DMRG solver parameters.
- `run_dmrg` returns `soe1` — the per-orbital entanglement measure.
- `EntropyConfig` + `select_by_entropy` filter the CAS down to the most
  correlated orbitals.
- The screened orbital lists `(screened_occ, screened_virt)` feed directly
  into the AO projection step.

**Next**: [03_ao_projection.ipynb](03_ao_projection.ipynb) — selecting orbitals by their chemical character using AO projection.